In [56]:
import os
import json
import requests

import pandas as pd
import dotenv
import redis

In [57]:
# open .env file and get API keys
env_path = os.path.abspath('../.env.development.local')
dotenv.load_dotenv(env_path)
KV_REST_API_READ_ONLY_TOKEN = os.getenv("KV_REST_API_READ_ONLY_TOKEN")
KV_REST_API_TOKEN = os.getenv("KV_REST_API_TOKEN")
KV_REST_API_URL = os.getenv("KV_REST_API_URL")
KV_URL = os.getenv("KV_URL")

# Set headers for authentication
headers = {
    "Authorization": f"Bearer {KV_REST_API_TOKEN}",
    "Content-Type": "application/json"
}

In [62]:
# Adjust url to work with redis
redis_url = KV_URL
if redis_url.startswith("redis://"):
    redis_url = 'rediss://' + redis_url[len('redis://'):]
r = redis.from_url(redis_url)

In [20]:
# Import datasets
datasets = {}
dataset_names = ['clfever', 'phemeplus', 'vitc']
for dataset_name in dataset_names:
    with open(f'{dataset_name}.json') as f:
        datasets[dataset_name] = json.load(f)

In [23]:
# Populate Vercel KV with datasets
for dataset_name in dataset_names:
    dataset = datasets[dataset_name]
    for datapoint in dataset:
        id = datapoint['claim_id']
        r.hset(id, mapping={
            'claim': datapoint['claim'],
            'evidence': datapoint['evidence'],
            'label': datapoint['label']
        })

In [21]:
# Create batches containing 25 datapoints each
batch_ids = []
for dataset_name in dataset_names:
    dataset = datasets[dataset_name]
    for i in range(0, len(dataset), 25):
        batch_dataset = dataset[i:i+25]
        claim_ids = []
        for batch_datapoint in batch_dataset:
            id = batch_datapoint['claim_id']
            claim_ids.append(id)
        batch_id = f'batch_{dataset_name}_{i//25 + 1}'
        batch_ids.append(batch_id)
        # r.hset(batch_id, mapping={'claim_ids': json.dumps(claim_ids)})    

In [49]:
# Create queue 
r.lpush('queue', *batch_ids)

69

In [44]:
queue = r.lrange('queue', 0, -1)
print(len(queue))
print(queue)

34
[b'batch_vitc_1', b'batch_phemeplus_4', b'batch_phemeplus_3', b'batch_phemeplus_2', b'batch_phemeplus_1', b'batch_clfever_1', b'batch_vitc_2', b'batch_vitc_1', b'batch_phemeplus_4', b'batch_phemeplus_3', b'batch_phemeplus_2', b'batch_phemeplus_1', b'batch_clfever_1', b'batch_vitc_2', b'batch_vitc_1', b'batch_phemeplus_4', b'batch_phemeplus_3', b'batch_phemeplus_2', b'batch_phemeplus_1', b'batch_clfever_1', b'batch_vitc_2', b'batch_vitc_1', b'batch_phemeplus_4', b'batch_phemeplus_3', b'batch_phemeplus_2', b'batch_phemeplus_1', b'batch_clfever_1', b'batch_vitc_2', b'batch_vitc_1', b'batch_phemeplus_4', b'batch_phemeplus_3', b'batch_phemeplus_2', b'batch_phemeplus_1', b'batch_clfever_1']


In [63]:
# Get list og participants who completed the task
r.lrange('participants',0,-1)

[b'{"participant":"123456","batchId":"batch_phemeplus_3"}',
 b'{"participant":"brave_academic","batchId":"batch_phemeplus_4"}',
 b'{"participant":"iphone-tester","batchId":"batch_vitc_1"}',
 b'{"participant":"Sad_Sylvanian","batchId":"batch_vitc_2"}',
 b'{"participant":"hello kitty","batchId":"batch_clfever_1"}',
 b'{"participant":"Test_kitten","batchId":"batch_vitc_1"}',
 b'{"participant":"Sebastian","batchId":"batch_phemeplus_3"}']

In [64]:
# get answers from specific participant
r.hgetall('123456')

{b'pplus_56': b'abductive',
 b'pplus_52': b'deductive',
 b'pplus_53': b'abductive',
 b'pplus_69': b'deductive',
 b'pplus_57': b'abductive',
 b'pplus_61': b'deductive',
 b'pplus_70': b'deductive',
 b'pplus_72': b'abductive',
 b'pplus_60': b'deductive',
 b'pplus_63': b'deductive',
 b'pplus_64': b'deductive',
 b'pplus_55': b'deductive',
 b'pplus_65': b'deductive',
 b'pplus_62': b'deductive',
 b'pplus_50': b'deductive',
 b'pplus_73': b'abductive',
 b'pplus_54': b'deductive',
 b'pplus_51': b'abductive',
 b'pplus_71': b'abductive',
 b'pplus_67': b'deductive',
 b'pplus_58': b'deductive',
 b'pplus_68': b'deductive',
 b'pplus_59': b'abductive',
 b'pplus_74': b'abductive',
 b'pplus_66': b'deductive'}